In [1]:
import pandas as pd
import duckdb as db

from models.TitleEmbeddingModel import TitleEmbeddingModel
from models.MFModel import MFModel
from models.RankingModel import RankingModel
from sentence_transformers import SentenceTransformer

/Users/logancostello/.pyenv/versions/spotify-playlist/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Data loading from evaluate model

groups = list(range(1, 11))

test_sets = []
train_sets = []
for i in groups:
    test_sets.append({
        'group': i,
        'playlist_metadata': pd.read_parquet(f"data/test/{i}/playlist_metadata.parquet"),
        'playlist_contents': pd.read_parquet(f"data/test/{i}/playlist_contents.parquet"),
        'holdout_contents':  pd.read_parquet(f"data/test/{i}/holdout_contents.parquet"),
    })
    train_sets.append({
        'group': i,
        'playlist_metadata': pd.read_parquet(f"data/train/{i}/playlist_metadata.parquet"),
        'playlist_contents': pd.read_parquet(f"data/train/{i}/playlist_contents.parquet"),
        'holdout_contents':  pd.read_parquet(f"data/train/{i}/holdout_contents.parquet"),
    })

ranker_train_metadata = pd.concat([ts['playlist_metadata'] for ts in train_sets], ignore_index=True)
ranker_train_contents = pd.concat([ts['playlist_contents'] for ts in train_sets], ignore_index=True)
ranker_train_holdouts = pd.concat([ts['holdout_contents']  for ts in train_sets], ignore_index=True)

original_metadata = pd.read_parquet("data/original/playlist_metadata.parquet")
original_contents = pd.read_parquet("data/original/playlist_contents.parquet")
track_metadata = pd.read_parquet("data/original/track_metadata.parquet")

train_pids = ranker_train_metadata["pid"].unique()
test_pids = pd.concat([ts['playlist_metadata']['pid'] for ts in test_sets]).unique()
held_pids  = set(train_pids) | set(test_pids)

candidate_train_metadata = pd.concat([
    original_metadata[~original_metadata["pid"].isin(held_pids)],
    ranker_train_metadata,
    *[ts['playlist_metadata'] for ts in test_sets],
], ignore_index=True)

candidate_train_contents = pd.concat([
    original_contents[~original_contents["pid"].isin(held_pids)],
    ranker_train_contents,
    *[ts['playlist_contents'] for ts in test_sets], 
], ignore_index=True)


del original_contents, original_metadata


In [3]:
# models
title_embedding_model = TitleEmbeddingModel()
mf_model = MFModel()
ranking_model = RankingModel(mf_model, title_embedding_model)

models = [title_embedding_model, mf_model, ranking_model]

In [4]:
# train models
for model in models:
    if model.is_ranker:
        model.train(ranker_train_metadata, ranker_train_contents, ranker_train_holdouts, track_metadata, candidate_train_metadata, candidate_train_contents)
    else:
        model.train(candidate_train_metadata, candidate_train_contents, track_metadata)

Save file found — loading TitleEmbedding model instead of retraining.
TitleEmbedding model loaded from saved_models/title_embedding_model/
Save file found — loading MF model instead of retraining.
MF model loaded from saved_models/mf_model/
Generating candidates for training...
Training on 11,260,455 candidates (655,611 positive)...
Training until validation scores don't improve for 25 rounds
[10]	valid_0's binary_logloss: 0.191409	valid_1's binary_logloss: 0.188888
[20]	valid_0's binary_logloss: 0.186164	valid_1's binary_logloss: 0.183745
[30]	valid_0's binary_logloss: 0.184324	valid_1's binary_logloss: 0.181988
[40]	valid_0's binary_logloss: 0.183539	valid_1's binary_logloss: 0.181279
[50]	valid_0's binary_logloss: 0.183115	valid_1's binary_logloss: 0.180923
[60]	valid_0's binary_logloss: 0.182851	valid_1's binary_logloss: 0.180728
[70]	valid_0's binary_logloss: 0.182645	valid_1's binary_logloss: 0.180585
[80]	valid_0's binary_logloss: 0.182481	valid_1's binary_logloss: 0.180493
[90]

In [ ]:
def build_custom_playlist(title, tracks, track_metadata, encoder, is_random_order=False, pid=-1):
    tm = track_metadata.copy()
    tm["track_name_lower"]  = tm["track_name"].str.lower()
    tm["artist_name_lower"] = tm["artist_name"].str.lower()

    matched_uris = []
    unmatched    = []

    for track_name, artist_name in tracks:
        mask = (
            tm["track_name_lower"].str.contains(track_name.lower(), regex=False) &
            tm["artist_name_lower"].str.contains(artist_name.lower(), regex=False)
        )
        hits = tm[mask]
        if hits.empty:
            unmatched.append((track_name, artist_name))
        else:
            matched_uris.append(hits.iloc[0]["track_uri"])

    if unmatched:
        print(f"No match found for {len(unmatched)} track(s):")
        for t, a in unmatched:
            print(f"  - '{t}' by '{a}'")

    print(title)
    for uri in matched_uris:
        row = tm[tm["track_uri"] == uri].iloc[0]
        print(f"  {row['track_name']} — {row['artist_name']}")

    embedding = encoder.encode(title)

    contents = pd.DataFrame({
        "pid":       pid,
        "track_uri": matched_uris,
        "position":  range(len(matched_uris)),
    })

    metadata = pd.DataFrame([{
        "pid":                   pid,
        "name":                  title,
        "num_tracks":            len(matched_uris),
        "title_bert_embeddings": embedding,
        "num_samples":           (matched_uris),
        "has_title":             title!="",
        "random_order":          is_random_order,
        "num_holdouts":          0,
    }])

    return metadata, contents

In [53]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

my_tracks = [
    ("The Killchain", "Bolt Thrower"),
]

my_metadata, my_contents = build_custom_playlist(
    title          = "",
    tracks         = my_tracks,
    track_metadata = track_metadata,
    encoder        = encoder,
    is_random_order= False,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5789.74it/s]



  The Killchain — Bolt Thrower


In [54]:
# title recs
recs = title_embedding_model.predict(my_metadata, my_contents, track_metadata, n_recs=25, g_num=-1)
recs.merge(track_metadata[["track_uri","track_name","artist_name"]], on="track_uri")[["track_name", "artist_name"]]


,track_name,artist_name
0,Broccoli (feat. Lil Yachty),DRAM
1,Needed Me,Rihanna
2,Hotline Bling,Drake
3,The Hills,The Weeknd
4,One Dance,Drake
5,Don't,Bryson Tiller
6,No Role Modelz,J. Cole
7,Black Beatles,Rae Sremmurd
8,Starboy,The Weeknd
9,Trap Queen,Fetty Wap


In [55]:
# mf recs
recs = mf_model.predict(my_metadata, my_contents, track_metadata, n_recs=25, g_num=-1)
recs.merge(track_metadata[["track_uri","track_name","artist_name"]], on="track_uri")[["track_name", "artist_name"]]


,track_name,artist_name
0,Midnight City,M83
1,Paper Planes,M.I.A.
2,Walking On A Dream,Empire of the Sun
3,Pumped Up Kicks,Foster The People
4,Electric Love,BØRNS
5,Breezeblocks,alt-J
6,Coffee,Sylvan Esso
7,She Will Be Loved - Radio Mix,Maroon 5
8,Wake Me Up - Radio Edit,Avicii
9,Gooey,Glass Animals


In [56]:
# ranker recs
recs = ranking_model.predict(my_metadata, my_contents, track_metadata, 25,-1,candidate_train_metadata,candidate_train_contents)
recs.merge(track_metadata[["track_uri","track_name","artist_name"]], on="track_uri")[["track_name", "artist_name"]]


,track_name,artist_name
0,Midnight City,M83
1,One Dance,Drake
2,Stressed Out,Twenty One Pilots
3,Paper Planes,M.I.A.
4,Needed Me,Rihanna
5,Breezeblocks,alt-J
6,Stolen Dance,Milky Chance
7,Do I Wanna Know?,Arctic Monkeys
8,Take Me To Church,Hozier
9,She Will Be Loved - Radio Mix,Maroon 5
